<a href="https://colab.research.google.com/github/Akshatha7710/RAG-based-Document-Question-Answering-System-using-LangChain-FAISS-and-LLMs/blob/main/Question%20Answering%20System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG-Based Document Question Answering System v2
**Stack:** PyMuPDF · ChromaDB (persistent) · Sentence-Transformers · Gemini 2.5 Flash


In [ ]:
# Cell 1: Install Dependencies
!pip install -q pymupdf chromadb sentence-transformers langchain-text-splitters google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 891.8 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [ ]:
# Cell 2: Set API Key
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("✅ API Key loaded:", os.environ["GEMINI_API_KEY"][:10], "...")

✅ API Key loaded: AIzaSyDFX_ ...


In [ ]:
# Cell 3: Imports & Setup
import fitz                          # PyMuPDF
import chromadb
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time
import os

# ── Gemini ────────────────────────────────────────────────────────────
genai.configure(api_key=os.environ["GEMINI_API_KEY"])
llm = genai.GenerativeModel("gemini-2.5-flash")

# ── Embedding model (local, no API needed) ────────────────────────────
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ── Persistent vector database ────────────────────────────────────────
# Data survives Colab runtime restarts as long as the /content folder is intact.
# Mount Google Drive first if you want true long-term persistence:
#   from google.colab import drive; drive.mount('/content/drive')
#   DB_PATH = "/content/drive/MyDrive/rag_chroma_db"
DB_PATH = "/content/chroma_db"
chroma = chromadb.PersistentClient(path=DB_PATH)
collection = chroma.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})

# ── Text splitter ─────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

print(f"✅ All components initialized")
print(f"📂 Vector DB path: {DB_PATH}")
print(f"📄 Documents already in collection: {collection.count()}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All components initialized
📂 Vector DB path: /content/chroma_db
📄 Documents already in collection: 0


In [ ]:
# Cell 4: Core Functions

# ── Retry helper ──────────────────────────────────────────────────────
def call_gemini(prompt: str, max_attempts: int = 5) -> str:
    """Call Gemini with exponential backoff on 503 / quota errors."""
    for attempt in range(max_attempts):
        try:
            response = llm.generate_content(prompt)
            return response.text
        except Exception as e:
            if attempt == max_attempts - 1:
                return f"❌ Failed after {max_attempts} attempts: {e}"
            wait = 2 ** attempt * 5   # 5s, 10s, 20s, 40s …
            print(f"⚠️  Attempt {attempt+1} failed — retrying in {wait}s… ({e})")
            time.sleep(wait)


# ── Ingestion (multi-document) ────────────────────────────────────────
def ingest_pdf(path: str):
    """
    Extract text from a PDF, chunk it, embed it, and store in ChromaDB.
    Each chunk is tagged with its source filename so answers can cite it.
    Safe to call multiple times — uses upsert with stable IDs.
    """
    filename = os.path.basename(path)
    doc  = fitz.open(path)
    text = "\n\n".join(page.get_text() for page in doc)
    doc.close()

    chunks     = splitter.split_text(text)
    embeddings = embedder.encode(chunks).tolist()
    # Stable IDs: <filename>_chunk_<index> avoids collisions across files
    ids        = [f"{filename}_chunk_{i}" for i in range(len(chunks))]
    metadatas  = [{"source": filename, "chunk_index": i} for i in range(len(chunks))]

    collection.upsert(ids=ids, embeddings=embeddings, documents=chunks, metadatas=metadatas)
    print(f"✅ Ingested {len(chunks)} chunks from '{filename}'")


def ingest_multiple_pdfs(paths: list):
    """Ingest a list of PDF file paths."""
    for path in paths:
        ingest_pdf(path)
    print(f"\n📚 Total chunks in DB: {collection.count()}")


# ── Single-turn QA ────────────────────────────────────────────────────
def ask(question: str, top_k: int = 3, show_sources: bool = True) -> str:
    """Retrieve relevant chunks and generate a grounded answer."""
    q_emb   = embedder.encode([question]).tolist()
    results = collection.query(
        query_embeddings=q_emb,
        n_results=top_k,
        include=["documents", "distances", "metadatas"]
    )
    chunks    = results["documents"][0]
    scores    = results["distances"][0]
    metadatas = results["metadatas"][0]
    context   = "\n\n---\n\n".join(chunks)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say \"I don't have enough information to answer that.\"

CONTEXT:
{context}

QUESTION:
{question}"""

    answer = call_gemini(prompt)

    if show_sources:
        print("─" * 60)
        print("📄 SOURCE CHUNKS USED:")
        for i, (chunk, score, meta) in enumerate(zip(chunks, scores, metadatas)):
            similarity = round((1 - score) * 100, 1)
            print(f"\n[Source {i+1}] — {similarity}% match | file: {meta.get('source', 'unknown')}")
            print(chunk[:300] + "..." if len(chunk) > 300 else chunk)
        print("─" * 60)

    return answer


# ── Conversational chat ───────────────────────────────────────────────
class RAGChat:
    """
    Stateful chat wrapper that keeps message history so follow-up
    questions ("Can you elaborate?", "What about X?") work correctly.
    """
    def __init__(self, top_k: int = 3, max_history_turns: int = 6):
        self.top_k = top_k
        self.max_history_turns = max_history_turns  # keep last N user+assistant pairs
        self.history: list[dict] = []               # [{role, content}, ...]

    def _retrieve(self, question: str) -> tuple[str, list, list, list]:
        q_emb   = embedder.encode([question]).tolist()
        results = collection.query(
            query_embeddings=q_emb,
            n_results=self.top_k,
            include=["documents", "distances", "metadatas"]
        )
        chunks    = results["documents"][0]
        scores    = results["distances"][0]
        metadatas = results["metadatas"][0]
        context   = "\n\n---\n\n".join(chunks)
        return context, chunks, scores, metadatas

    def chat(self, user_message: str, show_sources: bool = False) -> str:
        context, chunks, scores, metadatas = self._retrieve(user_message)

        # Build conversation history as a formatted string
        history_str = ""
        for turn in self.history[-(self.max_history_turns * 2):]:
            role = "User" if turn["role"] == "user" else "Assistant"
            history_str += f"{role}: {turn['content']}\n"

        prompt = f"""You are a helpful assistant. Answer using ONLY the context provided.
If the answer is not in the context, say \"I don't have enough information to answer that.\"
Use the conversation history to understand follow-up questions.

CONTEXT FROM DOCUMENTS:
{context}

CONVERSATION HISTORY:
{history_str}
User: {user_message}
Assistant:"""

        answer = call_gemini(prompt)

        # Save turn to history
        self.history.append({"role": "user",      "content": user_message})
        self.history.append({"role": "assistant", "content": answer})

        if show_sources:
            print("─" * 60)
            print("📄 SOURCE CHUNKS USED:")
            for i, (chunk, score, meta) in enumerate(zip(chunks, scores, metadatas)):
                similarity = round((1 - score) * 100, 1)
                print(f"[Source {i+1}] {similarity}% | {meta.get('source', 'unknown')}")
            print("─" * 60)

        return answer

    def reset(self):
        """Clear conversation history."""
        self.history = []
        print("🔄 Conversation history cleared.")


print("✅ Functions ready")

✅ Functions ready


In [ ]:
# Cell 5: Upload & Ingest One or More PDFs
from google.colab import files

uploaded = files.upload()   # opens file picker — select one or multiple PDFs
ingest_multiple_pdfs(list(uploaded.keys()))

Saving ai_basics.pdf to ai_basics.pdf
✅ Ingested 13 chunks from 'ai_basics.pdf'

📚 Total chunks in DB: 13


In [ ]:
# Cell 6: Single-Turn Q&A (original behaviour, now with source filenames)
questions = [
    "What are the three types of machine learning?",
    "How is AI used in healthcare?",
    "What are the ethical concerns around AI?",
    "What is the difference between deep learning and machine learning?"
]

for q in questions:
    print("\n" + "="*60)
    print(f"Q: {q}")
    print(ask(q))
    time.sleep(5)   # small buffer between calls


Q: What are the three types of machine learning?
────────────────────────────────────────────────────────────
📄 SOURCE CHUNKS USED:

[Source 1] — 74.0% match | file: ai_basics.pdf
Machine Learning (ML) is a subset of AI that enables systems to learn and improve from experience
without being explicitly programmed. Instead of writing rules manually, ML algorithms learn patterns
from data.
There are three main types of machine learning:
- Supervised Learning: The model is traine...

[Source 2] — 57.9% match | file: ai_basics.pdf
Introduction to Artificial Intelligence
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems.
These processes include learning, reasoning, and self-correction. AI has become one of the most
transformative technologies of the 21st century, impacting indus...

[Source 3] — 53.9% match | file: ai_basics.pdf
- Unsupervised Learning: The model finds hidden patterns in unlabeled data. Clustering customers by
purchasing beha

In [ ]:
# Cell 7: Ask Your Own Single Question
my_question = "Summarize the document in 3 bullet points"  # ← change this

print(f"Q: {my_question}")
print(ask(my_question))

Q: Summarize the document in 3 bullet points
────────────────────────────────────────────────────────────
📄 SOURCE CHUNKS USED:

[Source 1] — 14.4% match | file: ai_basics.pdf
modern large language models like GPT and Claude.
3. Natural Language Processing
Natural Language Processing (NLP) is a branch of AI focused on enabling computers to understand
and generate human language. Applications include machine translation, sentiment analysis, chatbots,
and text summarization...

[Source 2] — 10.6% match | file: ai_basics.pdf
government bodies worldwide are working to establish frameworks for ethical AI.
7. The Future of AI
The future of AI holds enormous promise. Artificial General Intelligence (AGI), which would match
human-level reasoning across all domains, remains a long-term research goal. In the near term, we can
...

[Source 3] — 7.4% match | file: ai_basics.pdf
transforming healthcare, finance, education, and many other sectors. While the opportunities are
immense, responsible dev

In [ ]:
# Cell 8: Conversational Chat Mode
# Follow-up questions now work because the full history is sent each turn.

bot = RAGChat(top_k=3)

# Example multi-turn conversation — edit freely
turns = [
    "What are the three types of machine learning?",
    "Can you give me a real-world example of the second one?",   # follow-up
    "What about the ethical concerns related to that?",          # follow-up
]

for turn in turns:
    print("\n" + "="*60)
    print(f"👤 You: {turn}")
    reply = bot.chat(turn, show_sources=False)
    print(f"🤖 Bot: {reply}")
    time.sleep(5)


👤 You: What are the three types of machine learning?
🤖 Bot: The three main types of machine learning are Supervised Learning, Unsupervised Learning, and Reinforcement Learning.

👤 You: Can you give me a real-world example of the second one?


⚠️  Attempt 1 failed — retrying in 5s… (429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 14.549596617s.)


⚠️  Attempt 2 failed — retrying in 10s… (429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 7.891667847s.)
🤖 Bot: Clustering customers by purchasing behavior.

👤 You: What about the ethical concerns related to that?
🤖 Bot: Key issues include algorithmic bias where AI systems can perpetuate or amplify existing societal biases, privacy concerns around data collection and surveillance, and the challenge of making AI systems explainable and transparent.


In [ ]:
# Cell 9: Evaluation — Semantic Similarity Scoring
# Measures how close the system's answers are to ground-truth answers
# using cosine similarity of sentence embeddings (0–1, higher = better).
# Replace the ground-truth answers below with ones from your own document.

import numpy as np

eval_set = [
    {
        "question": "What are the three types of machine learning?",
        "expected": "The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning."
    },
    {
        "question": "How is AI used in healthcare?",
        "expected": "AI is used in healthcare for predictive analytics to identify at-risk patients and for virtual health assistants that help manage chronic conditions."
    },
    {
        "question": "What is deep learning?",
        "expected": "Deep learning is a subset of machine learning that uses neural networks with many layers to model complex patterns."
    },
]

def semantic_similarity(a: str, b: str) -> float:
    """Cosine similarity between two sentences using the embedder."""
    vecs = embedder.encode([a, b])
    return float(np.dot(vecs[0], vecs[1]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1])))

scores = []
print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

for item in eval_set:
    predicted = ask(item["question"], show_sources=False)
    score     = semantic_similarity(predicted, item["expected"])
    scores.append(score)

    print(f"\nQ:        {item['question']}")
    print(f"Expected: {item['expected'][:120]}")
    print(f"Got:      {predicted[:120]}")
    print(f"Score:    {score:.3f}")
    time.sleep(5)

print("\n" + "=" * 60)
print(f"Average semantic similarity: {np.mean(scores):.3f}")
print("(1.0 = perfect match, >0.85 = very good, >0.70 = acceptable)")

EVALUATION RESULTS

Q:        What are the three types of machine learning?
Expected: The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.
Got:      The three main types of machine learning are:
- Supervised Learning
- Unsupervised Learning
- Reinforcement Learning
Score:    0.972

Q:        How is AI used in healthcare?
Expected: AI is used in healthcare for predictive analytics to identify at-risk patients and for virtual health assistants that he
Got:      Predictive analytics powered by AI can identify patients at risk of deterioration, enabling earlier intervention. Virtua
Score:    0.931

Q:        What is deep learning?
Expected: Deep learning is a subset of machine learning that uses neural networks with many layers to model complex patterns.
Got:      Deep Learning is a subset of machine learning that uses neural networks with many layers (hence deep) to model complex p
Score:    0.991

Average semantic similarity: 0.9